In [1]:
import pandas as pd 

# Childhood Obesity

### 1. Import Data

In [2]:
#MSOA11 LOOKUP
msoa11_lookup_df = pd.read_excel(open(r"N:\Geodatabase\Raw_Data\MSOA_(2011)_to_MSOA_(2021)_to_Local_Authority_District_(2022)_Lookup_for_England_and_Wales_7815823766405070629.xlsx", 'rb'), sheet_name='MSOA_(2011)_to_MSOA_(2021)__0') 

columns_to_keep = [
    'MSOA11CD', 'LAD22CD', 'LAD22NM'
]

#drop unwanted fields
msoa11_lookup_cleaned_df = msoa11_lookup_df[columns_to_keep]

#rename msoa11cd to make merging cleaner
msoa11_lookup_cleaned_df.rename(columns={
    'MSOA11CD': 'msoa11cd',
    'LAD22CD': 'lad22cd',
    'LAD22NM': 'lad22nm',
}, inplace=True)

msoa11_lookup_cleaned_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_25612\1953959756.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  msoa11_lookup_cleaned_df.rename(columns={


,msoa11cd,lad22cd,lad22nm
0,E02000001,E09000001,City of London
1,E02000002,E09000002,Barking and Dagenham
2,E02000003,E09000002,Barking and Dagenham
3,E02000004,E09000002,Barking and Dagenham
4,E02000005,E09000002,Barking and Dagenham


In [3]:
#OHID CHILDHOOD OBERSITY DATA PER MSOA
childhood_obesity_raw_data_df = pd.read_csv(r"N:\Geodatabase\Raw_Data\OHID_Childhood_Obesity.csv")
childhood_obesity_raw_data_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_25612\3004094222.py:2: DtypeWarning: Columns (20) have mixed types. Specify dtype option on import or set low_memory=False.
  childhood_obesity_raw_data_df = pd.read_csv(r"N:\Geodatabase\Raw_Data\OHID_Childhood_Obesity.csv")


,Indicator ID,Indicator Name,Parent Code,Parent Name,Area Code,Area Name,Area Type,Sex,Age,Category Type,...,Count,Denominator,Value note,Recent Trend,Compared to England value or percentiles,Compared to Counties & UAs (2021/22-2022/23) value or percentiles,Time period Sortable,New data,Compared to goal,Time period range
0,93105,Reception prevalence of obesity (including sev...,NaN,NaN,E92000001,England,England,Persons,4-5 yrs,NaN,...,151408.0,1573889.0,NaN,NaN,Not compared,Not compared,20080000,NaN,NaN,3y
1,93105,Reception prevalence of obesity (including sev...,E92000001,England,E06000001,Hartlepool,Counties & UAs (2021/22-2022/23),Persons,4-5 yrs,NaN,...,310.0,3115.0,NaN,NaN,Similar,Not compared,20080000,NaN,NaN,3y
2,93105,Reception prevalence of obesity (including sev...,E92000001,England,E06000002,Middlesbrough,Counties & UAs (2021/22-2022/23),Persons,4-5 yrs,NaN,...,510.0,5000.0,NaN,NaN,Similar,Not compared,20080000,NaN,NaN,3y
3,93105,Reception prevalence of obesity (including sev...,E92000001,England,E06000003,Redcar and Cleveland,Counties & UAs (2021/22-2022/23),Persons,4-5 yrs,NaN,...,405.0,4015.0,NaN,NaN,Similar,Not compared,20080000,NaN,NaN,3y
4,93105,Reception prevalence of obesity (including sev...,E92000001,England,E06000004,Stockton-on-Tees,Counties & UAs (2021/22-2022/23),Persons,4-5 yrs,NaN,...,665.0,6510.0,NaN,NaN,Similar,Not compared,20080000,NaN,NaN,3y


### 2. Clean up data and create a data frame for each indicator

In [5]:
# STEP 1: FILTER FOR ROWS WHERE 'AREA TYPE' EQUALS 'MSOA' and 'TIME PERIOD' EQUALS '2021/22 - 23/24'
childhood_obesity_df = childhood_obesity_raw_data_df[
    (childhood_obesity_raw_data_df['Area Type'] == 'MSOA') &
    (childhood_obesity_raw_data_df['Time period'] == '2021/22 - 23/24')
]

# STEP 2: KEEP ONLY THE SPECIFIED COLUMNS
columns_to_keep = [
    'Indicator Name', 'Parent Code', 'Parent Name', 'Area Code', 'Area Name',
    'Area Type', 'Time period', 'Value', 'Count', 'Denominator'
]

childhood_obesity_df = childhood_obesity_df[columns_to_keep]

# STEP 3: RENAME THE SPECIFIED COLUMNS
childhood_obesity_df.rename(columns={
    'Indicator Name': 'indicator_name',
    'Parent Code': 'country_code',
    'Parent Name': 'country_name',
    'Area Code': 'msoa11cd',
    'Area Name': 'msoa11nm',
    'Area Type': 'area_type',
    'Time period': 'time_period'
}, inplace=True)

# STEP 4: CREATE SEPARATE DATAFRAMES FOR EACH UNIQUE VALUE IN 'INDICATOR NAME'
unique_indicators = childhood_obesity_df['indicator_name'].unique()
dataframes = {}

for indicator in unique_indicators:
    dataframes[indicator] = childhood_obesity_df[childhood_obesity_df['indicator_name'] == indicator]

#STEP 5: CREATE INDIVIDUAL DATAFRAME FROM DICTIONARY
reception_prevalence_of_obesity_df = dataframes['Reception prevalence of obesity (including severe obesity), 3 years data combined']
reception_prevalence_of_overweight_df = dataframes['Reception prevalence of overweight (including obesity), 3 years data combined']
year_6_prevalence_of_obesity_df = dataframes['Year 6 prevalence of obesity (including severe obesity), 3 years data combined']
year_6_prevalence_of_overweight_df = dataframes['Year 6 prevalence of overweight (including obesity), 3 years data combined']

#STEP 6: RENAME VALUE, COUNT AND DENOMINATOR FIELDS
# Define renaming mappings for each DataFrame
rename_mappings = {
    "reception_prevalence_of_obesity_df": {
        'Value': 'reception_prevalence_of_obesity_perc',
        'Count': 'reception_prevalence_of_obesity_count',
        'Denominator': 'reception_prevalence_of_obesity_denominator',
    },
    "reception_prevalence_of_overweight_df": {
        'Value': 'reception_prevalence_of_overweight_perc',
        'Count': 'reception_prevalence_of_overweight_count',
        'Denominator': 'reception_prevalence_of_overweight_denominator',
    },
    "year_6_prevalence_of_obesity_df": {
        'Value': 'year_6_prevalence_of_obesity_perc',
        'Count': 'year_6_prevalence_of_obesity_count',
        'Denominator': 'year_6_prevalence_of_obesity_denominator',
    },
    "year_6_prevalence_of_overweight_df": {
        'Value': 'year_6_prevalence_of_overweight_perc',
        'Count': 'year_6_prevalence_of_overweight_count',
        'Denominator': 'year_6_prevalence_of_overweight_denominator',
    }
}

# Iterate over DataFrames and apply renaming
for df_name, renames in rename_mappings.items():
    locals()[df_name].rename(columns=renames, inplace=True)

# STEP 7: MERGE DATAFRAMES
# drop fields to avoid duplication
# list of columns to drop
columns_to_drop = [
    'indicator_name', 
    'country_code', 
    'country_name', 
    'msoa11nm', 
    'area_type', 
    'time_period'
]

# Drop the columns from the dataframe
reception_prevalence_of_obesity_df = reception_prevalence_of_obesity_df.drop(columns=['indicator_name'], errors='ignore')
reception_prevalence_of_overweight_df = reception_prevalence_of_overweight_df.drop(columns=columns_to_drop, errors='ignore')
year_6_prevalence_of_obesity_df = year_6_prevalence_of_obesity_df.drop(columns=columns_to_drop, errors='ignore')
year_6_prevalence_of_overweight_df = year_6_prevalence_of_overweight_df.drop(columns=columns_to_drop, errors='ignore')

#merge dataframes
step1_childhood_obesity_df = pd.merge(reception_prevalence_of_obesity_df,
                                      reception_prevalence_of_overweight_df, 
                                      how = 'left', 
                                      on = 'msoa11cd'
)
    
step2_childhood_obesity_df = pd.merge(step1_childhood_obesity_df,
                                      year_6_prevalence_of_obesity_df, 
                                      how = 'left', 
                                      on = 'msoa11cd'
)

step3_childhood_obesity_df = pd.merge(step2_childhood_obesity_df,
                                      year_6_prevalence_of_overweight_df, 
                                      how = 'left', 
                                      on = 'msoa11cd'
)

final_childhood_obesity_df = pd.merge(step3_childhood_obesity_df,
                                      msoa11_lookup_cleaned_df, 
                                      how = 'left', 
                                      on = 'msoa11cd'                                      
)


# Display the updated dataframe
final_childhood_obesity_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_25612\3444339594.py:66: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  locals()[df_name].rename(columns=renames, inplace=True)
C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_25612\3444339594.py:66: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  locals()[df_name].rename(columns=renames, inplace=True)
C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_25612\3444339594.py:66: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide

,country_code,country_name,msoa11cd,msoa11nm,area_type,time_period,reception_prevalence_of_obesity_perc,reception_prevalence_of_obesity_count,reception_prevalence_of_obesity_denominator,reception_prevalence_of_overweight_perc,reception_prevalence_of_overweight_count,reception_prevalence_of_overweight_denominator,year_6_prevalence_of_obesity_perc,year_6_prevalence_of_obesity_count,year_6_prevalence_of_obesity_denominator,year_6_prevalence_of_overweight_perc,year_6_prevalence_of_overweight_count,year_6_prevalence_of_overweight_denominator,lad22cd,lad22nm
0,E09000001,City of London,E02000001,City of London,MSOA,2021/22 - 23/24,NaN,NaN,NaN,26.66667,20.0,75.0,14.28571,10.0,70.0,35.71429,25.0,70.0,E09000001,City of London
1,E09000002,Barking and Dagenham,E02000002,Marks Gate,MSOA,2021/22 - 23/24,12.67606,45.0,355.0,28.16901,100.0,355.0,33.33333,135.0,405.0,48.14815,195.0,405.0,E09000002,Barking and Dagenham
2,E09000002,Barking and Dagenham,E02000003,Chadwell Heath East,MSOA,2021/22 - 23/24,10.30928,50.0,485.0,20.61856,100.0,485.0,29.67033,135.0,455.0,42.85714,195.0,455.0,E09000002,Barking and Dagenham
3,E09000002,Barking and Dagenham,E02000004,Eastbrookend,MSOA,2021/22 - 23/24,9.80392,25.0,255.0,21.56863,55.0,255.0,32.00000,80.0,250.0,44.00000,110.0,250.0,E09000002,Barking and Dagenham
4,E09000002,Barking and Dagenham,E02000005,Becontree Heath,MSOA,2021/22 - 23/24,12.62136,65.0,515.0,24.27184,125.0,515.0,31.06796,160.0,515.0,46.60194,240.0,515.0,E09000002,Barking and Dagenham


In [ ]:
year_6_prevalence_of_obesity_df